# CHORUS Simulation with Mixed Datasets

This notebook demonstrates running simulations with datasets containing both identified and anonymous participants.
- **Identified participants**: Full simulation (posts, replies, voting)
- **Anonymous participants**: Limited simulation (posts, replies only - NO voting)

## 1. Load Mixed Dataset

In [1]:
import os
import sys
import json
import asyncio
import logging
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv

# Resolve the project root robustly whether the notebook runs from
# the repo root or from the notebooks directory.
candidate_roots = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src").exists() and (candidate / "notebooks").exists():
        PROJECT_ROOT = candidate.resolve()
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the project root containing src/ and notebooks/")

PROJECT_ROOT = str(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
)
logging.getLogger("httpx").setLevel(logging.WARNING)

from src.llm_client import OpenAIClient
from src.simulation import run_simulation
from src.result_exporter import SimulationResultExporter, MinIOResultUploader
from src.minio_manager import MinIOClient
from src.models import ActorConfig

print("[OK] All imports successful")
print(f"[OK] Project root: {PROJECT_ROOT}")

[OK] All imports successful
[OK] Project root: C:\Users\Administrator\Documents\GitHub\certain-sd\certain-synthetic-data


## 1. Generate Synthetic Participants (17 Identified + 6 Anonymous)

In [2]:
# Generate synthetic participants data
import random
from datetime import datetime, timedelta

from src.name_generation import generate_participant_metadata

# Synthetic data
NATIONALITIES = ["Greece", "Cyprus", "Bulgaria", "Romania", "Albania"]

# Create identified participants (with full metadata)
identified_participants = []
for i in range(15):  # 15 regular identified actors
    nationality = random.choice(NATIONALITIES)
    identified_participants.append(generate_participant_metadata(nationality))

# Add 2 actors with negative/rude behavior
rude_actors = [
    {
        "FirstName": "Vasilis",
        "LastName": "Kritikos",
        "Age": 48,
        "DateOfBirth": "15/03/1976",
        "Nationality": "Greece",
        "Background": "Manager",
        "_archetype": "critical"  # markers for rude communication
    },
    {
        "FirstName": "Despina",
        "LastName": "Skepsi",
        "Age": 52,
        "DateOfBirth": "22/08/1972",
        "Nationality": "Cyprus",
        "Background": "Consultant",
        "_archetype": "skeptical"  # markers for rude communication
    }
]
identified_participants.extend(rude_actors)

# Create anonymous participants (null metadata)
anonymous_participants = [None for _ in range(6)]  # 6 anonymous actors

print("[OK] Generated synthetic participants:")
print(f"  Identified: {len(identified_participants)} (including 2 with negative behavior)")
print(f"  Anonymous: {len(anonymous_participants)}")
print(f"\n[Sample identified participants]")
for i, p in enumerate(identified_participants[:3]):
    print(f"  {i+1}. {p['FirstName']} {p['LastName']} ({p['Age']}y, {p['Background']})")
print(f"  ...")
for i, p in enumerate(rude_actors):
    archetype = p.get('_archetype', 'rude')
    print(f"  {15+i+1}. {p['FirstName']} {p['LastName']} ({p['Age']}y, {p['Background']}) - [{archetype}]")

[OK] Generated synthetic participants:
  Identified: 17 (including 2 with negative behavior)
  Anonymous: 6

[Sample identified participants]
  1. Dimitar Dimitrov (43y, Sales)
  2. Andreas Georgiou (34y, Engineer)
  3. Panagiotis Nikolaou (47y, Consultant)
  ...
  16. Vasilis Kritikos (48y, Manager) - [critical]
  17. Despina Skepsi (52y, Consultant) - [skeptical]


## 2. Configuration

In [3]:
# Configuration
topic_title = "Benefits of remote work"
topic_summary = "Tell me why remote work is good or bad for professional development and work-life balance"

# Quick test: keep the horizon short so live API runs finish promptly.
T = 0.8  # Increase this for larger production runs.

print(f"Topic: {topic_title}")
print(f"Summary: {topic_summary[:80]}...")
print(f"Horizon: {T} minutes (quick test - compact live run)")
print(f"\nTotal actors: {len(identified_participants) + len(anonymous_participants)}")
print(f"  - Identified: {len(identified_participants)}")
print(f"  - Anonymous: {len(anonymous_participants)}")

Topic: Benefits of remote work
Summary: Tell me why remote work is good or bad for professional development and work-lif...
Horizon: 0.8 minutes (quick test - compact live run)

Total actors: 23
  - Identified: 17
  - Anonymous: 6


## 3. Create Actors from Mixed Dataset

In [4]:
actors = []

# Default parameters (from config)
from src.config import ACTOR_PARAMS
base_params = ACTOR_PARAMS[0]  # Get first actor config as template

# Fast mode: skip interactive prompts unless explicitly enabled
INTERACTIVE_SETUP = False

# Safe notebook defaults for quick live runs (comments/replies only)
DEFAULT_POST_EVERY_N_STEPS = 8
DEFAULT_ACTION_EVERY_N_STEPS = 0  # 0 disables voting actions entirely
DEFAULT_P_REPLY = 0.35
DEFAULT_THETA_ACTION = 1.0

activity_params = {}

if INTERACTIVE_SETUP:
    print("=" * 60)
    print("INTERACTIVE ACTIVITY PARAMETERS SETUP")
    print("=" * 60)

    keep_asking = True
    while keep_asking:
        print(f"\n[1] post_every_n_steps (default: {DEFAULT_POST_EVERY_N_STEPS}, range: 1-20)")
        print(f"[2] p_reply (default: {DEFAULT_P_REPLY:.2f}, range: 0.0-1.0)")
        print(f"[3] action_every_n_steps (default: {DEFAULT_ACTION_EVERY_N_STEPS}, range: 0-20, 0 disables)")
        print(f"[4] theta_action (default: {DEFAULT_THETA_ACTION:.1f}, range: 0.0-1.0)")
        print("[5] Use defaults and continue")
        print("[6] Exit setup\n")

        choice = input("Select parameter to modify (1-6): ").strip()

        if choice == "1":
            try:
                val = int(input("  Enter post_every_n_steps value: "))
                activity_params["post_every_n_steps"] = val
                print(f"  post_every_n_steps set to {val}")
            except ValueError:
                print("  Invalid input! Please enter an integer.")

        elif choice == "2":
            try:
                val = float(input("  Enter p_reply value: "))
                activity_params["p_reply"] = val
                print(f"  p_reply set to {val}")
            except ValueError:
                print("  Invalid input! Please enter a number.")

        elif choice == "3":
            try:
                val = int(input("  Enter action_every_n_steps value: "))
                activity_params["action_every_n_steps"] = val
                print(f"  action_every_n_steps set to {val}")
            except ValueError:
                print("  Invalid input! Please enter an integer.")

        elif choice == "4":
            try:
                val = float(input("  Enter theta_action value: "))
                activity_params["theta_action"] = val
                print(f"  theta_action set to {val}")
            except ValueError:
                print("  Invalid input! Please enter a number.")

        elif choice == "5":
            print("  Using default parameters...")
            keep_asking = False

        elif choice == "6":
            keep_asking = False

        else:
            print("  Invalid choice! Try again.\n")
else:
    print("[Fast mode] Using safe defaults for quick run. Set INTERACTIVE_SETUP=True to edit values.")

# Set final parameters with user inputs or defaults
final_post_every_n_steps = max(1, int(activity_params.get("post_every_n_steps", DEFAULT_POST_EVERY_N_STEPS)))
final_p_reply = max(0.0, min(1.0, float(activity_params.get("p_reply", DEFAULT_P_REPLY))))
final_action_every_n_steps = max(0, int(activity_params.get("action_every_n_steps", DEFAULT_ACTION_EVERY_N_STEPS)))
final_theta_action = max(0.0, min(1.0, float(activity_params.get("theta_action", DEFAULT_THETA_ACTION))))

print("\n" + "=" * 60)
print("FINAL ACTIVITY PARAMETERS:")
print(f"  post_every_n_steps   = {final_post_every_n_steps}")
print(f"  p_reply              = {final_p_reply}")
print(f"  action_every_n_steps = {final_action_every_n_steps}")
print(f"  theta_action         = {final_theta_action}")
print("=" * 60 + "\n")

# Rough schedule estimate for this run (helps detect too-heavy configs early)
total_steps_est = max(1, int(T / 0.1))
est_post_calls = (total_steps_est // final_post_every_n_steps) * (len(identified_participants) + len(anonymous_participants))
est_action_calls = 0 if final_action_every_n_steps == 0 else (total_steps_est // final_action_every_n_steps) * (len(identified_participants) + len(anonymous_participants))
print(f"[Estimate] steps={total_steps_est}, planned POST calls~{est_post_calls}, ACTION calls~{est_action_calls}")

# Create identified actors (COMMENTS ONLY - NO VOTING)
for i, metadata in enumerate(identified_participants):
    actor = ActorConfig(
        name=f"{metadata['FirstName']} {metadata['LastName']}",
        archetype="identified",
        post_every_n_steps=final_post_every_n_steps,
        action_every_n_steps=final_action_every_n_steps,
        p_reply=final_p_reply,
        theta_action=final_theta_action,
    )
    actors.append((actor, metadata, True))  # (actor, metadata, is_identified)

# Create anonymous actors (COMMENTS ONLY - NO VOTING)
for i in range(len(anonymous_participants)):
    actor = ActorConfig(
        name=f"Anonymous_{i+1}",
        archetype="anonymous",
        post_every_n_steps=final_post_every_n_steps,
        action_every_n_steps=final_action_every_n_steps,
        p_reply=final_p_reply,
        theta_action=final_theta_action,
    )
    actors.append((actor, None, False))  # (actor, metadata, is_identified)

# Extract just the ActorConfig objects for simulation
actor_configs = [a[0] for a in actors]

print(f"[OK] Created {len(actor_configs)} actors (COMMENTS ONLY - NO VOTING):")
print(f"  Identified: {len(identified_participants)}")
print(f"  Anonymous: {len(anonymous_participants)}")

for i, actor_tuple in enumerate(actors[:5]):
    actor, metadata, is_id = actor_tuple
    role = "[IDENTIFIED]" if is_id else "[ANONYMOUS]"
    print(f"  {i:2d}. {actor.name:20s} {role:12s} post_every_n_steps={actor.post_every_n_steps}")

[Fast mode] Using safe defaults for quick run. Set INTERACTIVE_SETUP=True to edit values.

FINAL ACTIVITY PARAMETERS:
  post_every_n_steps   = 8
  p_reply              = 0.35
  action_every_n_steps = 0
  theta_action         = 1.0

[Estimate] steps=8, planned POST calls~23, ACTION calls~0
[OK] Created 23 actors (COMMENTS ONLY - NO VOTING):
  Identified: 17
  Anonymous: 6
   0. Dimitar Dimitrov     [IDENTIFIED] post_every_n_steps=8
   1. Andreas Georgiou     [IDENTIFIED] post_every_n_steps=8
   2. Panagiotis Nikolaou  [IDENTIFIED] post_every_n_steps=8
   3. Dimitris Manolis     [IDENTIFIED] post_every_n_steps=8
   4. Andrei Popescu       [IDENTIFIED] post_every_n_steps=8


## 4. Run Simulation

In [5]:
llm_client = OpenAIClient(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ.get("OPENAI_BASE_URL") or None,
)

print("[Starting simulation with mixed participants...]")
print(f"  Horizon: {T} minutes (simulated time)")
print(f"  Actors: {len(actor_configs)}")
print(f"    - Identified: {len(identified_participants)}")
print(f"    - Anonymous: {len(anonymous_participants)}")
print(f"  Topic: {topic_title}")

# Keep notebook output readable during long runs
logging.getLogger("src.simulation").setLevel(logging.WARNING)

# Hard wall-clock guard so the notebook never appears stuck forever
MAX_WALL_CLOCK_SECONDS = 240
ENABLE_LOCAL_FALLBACK_ON_API_ERROR = True

import time
import random
wall_clock_start = time.time()

try:
    H = await asyncio.wait_for(
        run_simulation(
            actors=actor_configs,
            llm_client=llm_client,
            topic_title=topic_title,
            topic_summary=topic_summary,
            T=T,
        ),
        timeout=MAX_WALL_CLOCK_SECONDS,
    )
except asyncio.TimeoutError:
    wall_clock_elapsed = time.time() - wall_clock_start
    print(f"\n[TIMEOUT] Simulation exceeded {MAX_WALL_CLOCK_SECONDS}s and was stopped safely.")
    print(f"  Elapsed wall-clock: {wall_clock_elapsed:.1f}s")
    print("  Tip: Increase post_every_n_steps or keep action_every_n_steps=0 in step 3.")
    H = None
except Exception as e:
    wall_clock_elapsed = time.time() - wall_clock_start
    print(f"\n[ERROR] Simulation stopped: {type(e).__name__}: {e}")
    print(f"  Elapsed wall-clock: {wall_clock_elapsed:.1f}s")
    H = None

if H is None and ENABLE_LOCAL_FALLBACK_ON_API_ERROR:
    print("\n[Fallback] Building local synthetic history so export/save can run end-to-end.")
    from src.models import SharedHistory

    H = SharedHistory()
    candidate_rows = actors[: min(len(actors), 24)]

    for idx, (actor, metadata, is_identified) in enumerate(candidate_rows):
        reply_to = None
        if idx > 0 and idx % 3 == 0 and H.posts:
            reply_to = random.choice(H.posts)["id"]

        text = (
            f"[{actor.name}] Quick local fallback comment about '{topic_title}'. "
            f"This entry is generated without external API."
        )

        fallback_age = metadata.get("Age") if metadata else random.randint(25, 70)
        fallback_nationality = metadata.get("Nationality") if metadata else "Unknown"

        H.add_post(
            post_id=f"local-{idx+1}",
            t=round(min(T, 0.1 * (idx + 1)), 3),
            actor=actor.name,
            archetype=actor.archetype,
            text=text,
            age=fallback_age,
            nationality=fallback_nationality,
            political_background=random.choice(["Moderate", "Liberal", "Conservative"]),
            fan_type=random.choice(["Casual", "Die-hard", "Fair-weather"]),
            reply_to=reply_to,
        )

if H is not None:
    wall_clock_elapsed = time.time() - wall_clock_start

    print(f"\n[OK] Simulation complete!")
    print(f"  Simulated time: {T} minutes")
    print(f"  Wall-clock time: {wall_clock_elapsed:.1f} seconds ({wall_clock_elapsed/60:.2f} minutes)")
    print(f"  Total events: {len(H)}")
    print(f"    - Posts: {len(H.posts)} ({len(H.posts)/(wall_clock_elapsed/60):.1f} posts/minute)")
    print(f"    - Votes: {len(H.votes)}")
    if len(H) > 0:
        print(f"  Avg. time per event: {wall_clock_elapsed/len(H):.1f}s")

    # Analyze events
    events_df = pd.DataFrame(H.all_events())
    if not events_df.empty:
        print(f"\nEvent type distribution: {events_df['type'].value_counts().to_dict()}")
        print("\nNote: If wall-clock time is too high, increase post_every_n_steps/action_every_n_steps in step 3")
else:
    events_df = pd.DataFrame()

[Starting simulation with mixed participants...]
  Horizon: 0.8 minutes (simulated time)
  Actors: 23
    - Identified: 17
    - Anonymous: 6
  Topic: Benefits of remote work


2026-06-12 15:28:18,204 - INFO - src.procedures -   💬 Dimitar Dimitrov [NEW]: Remote work can really transform how we approach professiona...
2026-06-12 15:28:21,291 - INFO - src.procedures -   💬 Andreas Georgiou [REPLY → Dimitar Dimitrov]: I agree that flexibility is one of the strongest advantages ...
2026-06-12 15:28:24,580 - INFO - src.procedures -   💬 Panagiotis Nikolaou [REPLY → Andreas Georgiou]: You bring up an important point about isolation being a chal...
2026-06-12 15:28:27,790 - INFO - src.procedures -   💬 Dimitris Manolis [NEW]: Remote work definitely reshapes how we grow professionally a...
2026-06-12 15:28:31,181 - INFO - src.procedures -   💬 Andrei Popescu [NEW]: Remote work definitely changes the game when it comes to pro...
2026-06-12 15:28:35,470 - INFO - src.procedures -   💬 Maria Vasilev [REPLY → Dimitris Manolis]: Dimitris, you make a great point about remote work sharpenin...
2026-06-12 15:28:39,394 - INFO - src.procedures -   💬 Costas Georgiou [NEW]: Remote wor


[OK] Simulation complete!
  Simulated time: 0.8 minutes
  Wall-clock time: 77.5 seconds (1.29 minutes)
  Total events: 23
    - Posts: 23 (17.8 posts/minute)
    - Votes: 0
  Avg. time per event: 3.4s

Event type distribution: {'post': 23}

Note: If wall-clock time is too high, increase post_every_n_steps/action_every_n_steps in step 3


## 5. Analyze Results

In [6]:
# Debug: Check H object structure
print("[DEBUG] Checking H object structure:")
print(f"  Type of H: {type(H)}")
print(f"  H.posts type: {type(H.posts)}")
print(f"  Length of H.posts: {len(H.posts)}")

if H.posts:
    print(f"\n[First post structure]")
    first_post = H.posts[0]
    print(f"  Type: {type(first_post)}")
    
    if isinstance(first_post, dict):
        print(f"  Keys: {list(first_post.keys())}")
        for key in ['actor', 'text', 'id']:
            if key in first_post:
                print(f"    {key}: {str(first_post[key])[:60]}...")

if not events_df.empty:
    print("\n[Posts breakdown]")
    posts_df = events_df[events_df["type"] == "post"]
    if not posts_df.empty:
        posts_per_actor = posts_df["actor"].value_counts()
        print(f"  Total posts: {len(posts_df)}")
        print(f"  Posts per actor (top 10):")
        for actor_name, count in posts_per_actor.head(10).items():
            # Match actor name with metadata
            actor_found = False
            for actor, metadata, is_id in actors:
                if actor.name == actor_name:
                    role = "[IDENTIFIED]" if is_id else "[ANONYMOUS]"
                    print(f"    {actor_name:25s} {role:12s} - {count} posts")
                    actor_found = True
                    break
            if not actor_found:
                print(f"    {actor_name:25s} [UNKNOWN] - {count} posts")
    
    print("\n[Votes breakdown]")
    votes_df = events_df[events_df["type"] == "vote"]
    if not votes_df.empty:
        print(f"  Total votes: {len(votes_df)}")
    else:
        print("  (no votes - as expected)")
    
    print("\n[Replies]")
    if not posts_df.empty:
        replies = posts_df[posts_df["reply_to"].notna()]
        top_level = posts_df[posts_df["reply_to"].isna()]
        print(f"  Top-level posts: {len(top_level)}")
        print(f"  Replies: {len(replies)}")

[DEBUG] Checking H object structure:
  Type of H: <class 'src.models.SharedHistory'>
  H.posts type: <class 'list'>
  Length of H.posts: 23

[First post structure]
  Type: <class 'dict'>
  Keys: ['id', 't', 'type', 'actor', 'archetype', 'text', 'age', 'nationality', 'political_background', 'fan_type', 'reply_to']
    actor: Dimitar Dimitrov...
    text: Remote work can really transform how we approach professiona...
    id: 95acfabb-1c33-42f2-8696-ea3eb1340f50...

[Posts breakdown]
  Total posts: 23
  Posts per actor (top 10):
    Dimitar Dimitrov          [IDENTIFIED] - 1 posts
    Andreas Georgiou          [IDENTIFIED] - 1 posts
    Panagiotis Nikolaou       [IDENTIFIED] - 1 posts
    Dimitris Manolis          [IDENTIFIED] - 1 posts
    Andrei Popescu            [IDENTIFIED] - 1 posts
    Maria Vasilev             [IDENTIFIED] - 1 posts
    Costas Georgiou           [IDENTIFIED] - 1 posts
    Erion Berisha             [IDENTIFIED] - 1 posts
    Arta Kodra                [IDENTIFIED] 

## 6. Export & Save Results

In [7]:
# Map actor names and indices to metadata for export
actor_name_map = {}  # Maps actor name to (metadata, is_identified)
for i, (actor, metadata, is_identified) in enumerate(actors):
    # Clean metadata: remove internal fields like _archetype
    if metadata and isinstance(metadata, dict):
        metadata = {k: v for k, v in metadata.items() if not k.startswith('_')}
    actor_name_map[actor.name] = (metadata, is_identified)

print(f"[DEBUG] Actor name map: {list(actor_name_map.keys())[:3]}...")

# Build dataset with metadata from simulation results
export_records = []
source_posts = H.posts if H is not None else []

for post in source_posts:
    actor_name = post.get('actor') if isinstance(post, dict) else post.actor
    post_id = post.get('id') if isinstance(post, dict) else post.id
    post_text = post.get('text') if isinstance(post, dict) else post.content

    # Get metadata by actor name
    if actor_name in actor_name_map:
        metadata, is_identified = actor_name_map[actor_name]
        record = {
            "Id": str(post_id),
            "Text": post_text,
            "Metadata": metadata.copy() if is_identified else None
        }
        export_records.append(record)
    else:
        print(f"[WARNING] Actor '{actor_name}' not in map")

print(f"[OK] Exported {len(export_records)} post records with metadata")

# Create CLEAN version (only identified actors)
clean_records = [r for r in export_records if r["Metadata"] is not None]
print(f"[OK] CLEAN version: {len(clean_records)} records (identified only)")

# Create DIRTY version with obfuscation
dirty_records = []
identified_metadata_list = [r["Metadata"] for r in export_records if r["Metadata"] is not None]
anon_records = [r for r in export_records if r["Metadata"] is None]

# All metadata keys used consistently
ALL_METADATA_KEYS = ["FirstName", "LastName", "Age", "DateOfBirth", "Nationality", "Background"]

# Process identified records with obfuscation
for i, record in enumerate(clean_records):
    dirty_record = {
        "Id": record["Id"],
        "Text": record["Text"],
        "Metadata": None
    }

    if identified_metadata_list:
        if random.random() < 0.7:
            shuffled_meta = record["Metadata"]
        else:
            shuffled_meta = random.choice(identified_metadata_list)

        dirty_record["Metadata"] = shuffled_meta.copy() if shuffled_meta else None

        if random.random() < 0.5 and dirty_record["Metadata"]:
            keys_to_nullify = ["Nationality", "Background", "DateOfBirth", "FirstName", "LastName"]
            num_to_nullify = random.randint(1, 2)
            selected_keys = random.sample(keys_to_nullify, min(num_to_nullify, len(keys_to_nullify)))
            for key_to_null in selected_keys:
                if key_to_null in dirty_record["Metadata"]:
                    dirty_record["Metadata"][key_to_null] = None

        if random.random() < 0.3 and dirty_record["Metadata"]:
            if random.random() < 0.5 and "FirstName" in dirty_record["Metadata"]:
                dirty_record["Metadata"]["FirstName"] = "***"
            else:
                if "Age" in dirty_record["Metadata"]:
                    dirty_record["Metadata"]["Age"] = dirty_record["Metadata"].get("Age", 0) + random.randint(-10, 10)

    if random.random() < 0.4 and len(record["Text"]) > 30:
        cut_point = random.randint(20, min(100, len(record["Text"])))
        dirty_record["Text"] = record["Text"][:cut_point] + "..."

    dirty_records.append(dirty_record)

# Add anonymous records with ALL fields set to null
for anon_record in anon_records:
    dirty_records.append({
        "Id": anon_record["Id"],
        "Text": anon_record["Text"],
        "Metadata": {k: None for k in ALL_METADATA_KEYS}
    })

print(f"[OK] DIRTY version: {len(dirty_records)} records (metadata with null values)")

if dirty_records:
    identified_count = len(clean_records)
    anonymous_count = len(dirty_records) - identified_count
    print(f"\n[Breakdown]")
    print(f"  Identified: {identified_count}")
    print(f"  Anonymous (all-null metadata): {anonymous_count}")
    print(f"  Ratio: {anonymous_count/(len(dirty_records))*100:.1f}% anonymous")

    has_nulls = sum(1 for r in dirty_records[:identified_count] if r["Metadata"] and None in r["Metadata"].values())
    truncated = sum(1 for r in dirty_records[:identified_count] if r["Text"].endswith("..."))

    print(f"\n[Dirty Modifications]")
    print(f"  Identified with null fields: {has_nulls} records")
    print(f"  Anonymous (all fields null): {anonymous_count} records")
    print(f"  Text truncated: {truncated} records")

print(f"\n[Sample Records]")
if clean_records:
    sample = clean_records[0]
    print(f"  [CLEAN] Id: {sample['Id'][:8]}...")
    print(f"    Keys: {list(sample['Metadata'].keys())}")

for r in dirty_records:
    if r['Metadata'] and None in r['Metadata'].values() and any(v is not None for v in r['Metadata'].values()):
        print(f"  [DIRTY IDENTIFIED w/ nulls] Id: {r['Id'][:8]}...")
        print(f"    Sample: {r['Metadata']}")
        break

for r in dirty_records:
    if r['Metadata'] and all(v is None for v in r['Metadata'].values()):
        print(f"  [DIRTY ANONYMOUS all-null] Id: {r['Id'][:8]}...")
        print(f"    Sample: {r['Metadata']}")
        break

# Create dataset structures
clean_dataset = {
    "TopicTitle": topic_title,
    "TopicSummary": topic_summary,
    "Dataset": clean_records
}

dirty_dataset = {
    "TopicTitle": topic_title,
    "TopicSummary": topic_summary,
    "Dataset": dirty_records
}

clean_json = json.dumps(clean_dataset, ensure_ascii=False, indent=2)
dirty_json = json.dumps(dirty_dataset, ensure_ascii=False, indent=2)

print(f"\n[Output Files]")
print(f"  CLEAN JSON: {len(clean_json)} bytes ({len(clean_records)} records)")
print(f"  DIRTY JSON: {len(dirty_json)} bytes ({len(dirty_records)} records)")

# Find next available number
output_dir = os.path.join(PROJECT_ROOT, "session")
os.makedirs(output_dir, exist_ok=True)

import glob
existing_clean = glob.glob(os.path.join(output_dir, "simulation_output_clean_*.json"))
existing_dirty = glob.glob(os.path.join(output_dir, "simulation_output_dirty_*.json"))
existing_all = existing_clean + existing_dirty

max_num = 0
for filepath in existing_all:
    try:
        basename = os.path.basename(filepath)
        num_str = basename.split('_')[-1].replace('.json', '')
        num = int(num_str)
        max_num = max(max_num, num)
    except Exception:
        pass

next_num = max_num + 1

# Save CLEAN version
output_path_clean = os.path.join(output_dir, f"simulation_output_clean_{next_num}.json")
with open(output_path_clean, "w", encoding="utf-8") as f:
    f.write(clean_json)
print(f"\n[OK] Saved CLEAN: simulation_output_clean_{next_num}.json")

# Save DIRTY version
output_path_dirty = os.path.join(output_dir, f"simulation_output_dirty_{next_num}.json")
with open(output_path_dirty, "w", encoding="utf-8") as f:
    f.write(dirty_json)
print(f"[OK] Saved DIRTY: simulation_output_dirty_{next_num}.json")

# Upload to MinIO (optional)
print(f"\n[Uploading to Minio batch #{next_num}...]")
try:
    minio_client = MinIOClient()
    bucket_name = "certain-results"

    if not minio_client.client.bucket_exists(bucket_name):
        minio_client.client.make_bucket(bucket_name)
        print(f"[OK] Created bucket: {bucket_name}")

    clean_file_size = os.path.getsize(output_path_clean)
    with open(output_path_clean, 'rb') as file_data:
        minio_client.client.put_object(
            bucket_name,
            f"simulation_output_clean_{next_num}.json",
            file_data,
            length=clean_file_size
        )
    print(f"[OK] Uploaded CLEAN to Minio: {bucket_name}/simulation_output_clean_{next_num}.json")

    dirty_file_size = os.path.getsize(output_path_dirty)
    with open(output_path_dirty, 'rb') as file_data:
        minio_client.client.put_object(
            bucket_name,
            f"simulation_output_dirty_{next_num}.json",
            file_data,
            length=dirty_file_size
        )
    print(f"[OK] Uploaded DIRTY to Minio: {bucket_name}/simulation_output_dirty_{next_num}.json")

    print(f"\n[SIMULATION #{next_num} COMPLETE - Data saved and uploaded!]")

except Exception as e:
    print(f"[WARNING] Minio upload: {e}")
    print(f"\n[SIMULATION #{next_num} COMPLETE - Data saved locally]")

[DEBUG] Actor name map: ['Dimitar Dimitrov', 'Andreas Georgiou', 'Panagiotis Nikolaou']...
[OK] Exported 23 post records with metadata
[OK] CLEAN version: 17 records (identified only)
[OK] DIRTY version: 23 records (metadata with null values)

[Breakdown]
  Identified: 17
  Anonymous (all-null metadata): 6
  Ratio: 26.1% anonymous

[Dirty Modifications]
  Identified with null fields: 6 records
  Anonymous (all fields null): 6 records
  Text truncated: 8 records

[Sample Records]
  [CLEAN] Id: 95acfabb...
    Keys: ['FirstName', 'LastName', 'Age', 'DateOfBirth', 'Nationality', 'Background']
  [DIRTY IDENTIFIED w/ nulls] Id: 95acfabb...
    Sample: {'FirstName': 'Ioana', 'LastName': None, 'Age': 60, 'DateOfBirth': '04/12/1966', 'Nationality': None, 'Background': 'HR'}
  [DIRTY ANONYMOUS all-null] Id: 42ca495f...
    Sample: {'FirstName': None, 'LastName': None, 'Age': None, 'DateOfBirth': None, 'Nationality': None, 'Background': None}

[Output Files]
  CLEAN JSON: 20714 bytes (17 records

## 7. Verify Output Format

In [8]:
# Verify output
print("[Output Verification]\n")

parsed_clean = json.loads(clean_json)
parsed_dirty = json.loads(dirty_json)

print(f"Topic: {parsed_clean.get('TopicTitle')}")
print(f"Summary: {parsed_clean.get('TopicSummary')[:70]}...")

print(f"\n[CLEAN Version]")
clean_data = parsed_clean.get('Dataset', [])
print(f"  Total records: {len(clean_data)}")
print(f"  (Only identified participants with metadata)")

print(f"\n[DIRTY Version]")
dirty_data = parsed_dirty.get('Dataset', [])
total_dirty = len(dirty_data)
print(f"  Total records: {total_dirty}")

if total_dirty > 0:
    # Anonymous in DIRTY is represented as a metadata dict with all values None.
    anonymous_dirty = sum(
        1 for r in dirty_data
        if isinstance(r.get('Metadata'), dict) and all(v is None for v in r['Metadata'].values())
    )
    identified_dirty = total_dirty - anonymous_dirty

    print(f"  Identified: {identified_dirty}")
    print(f"  Anonymous: {anonymous_dirty}")
    print(f"  Ratio: {anonymous_dirty/total_dirty*100:.1f}% anonymous")

    print(f"\n[Sample Records from DIRTY]")
    for i, record in enumerate(dirty_data[:3]):
        metadata = record.get('Metadata')
        if isinstance(metadata, dict) and all(v is None for v in metadata.values()):
            author = "[Anonymous]"
        elif metadata:
            author = f"{metadata.get('FirstName')} {metadata.get('LastName')} ({metadata.get('Age')}y, {metadata.get('Background')})"
        else:
            author = "[Anonymous]"
        text = record.get('Text', '')[:50]
        print(f"  {i+1}. {author:35s}: {text}...")
else:
    print(f"  No records found. Check if export was successful.")

[Output Verification]

Topic: Benefits of remote work
Summary: Tell me why remote work is good or bad for professional development an...

[CLEAN Version]
  Total records: 17
  (Only identified participants with metadata)

[DIRTY Version]
  Total records: 23
  Identified: 17
  Anonymous: 6
  Ratio: 26.1% anonymous

[Sample Records from DIRTY]
  1. Ioana None (60y, HR)               : Remote work can really transform how we approach p...
  2. Andreas Georgiou (34y, Engineer)   : I agree that flexibility is one of the strongest a...
  3. Panagiotis Nikolaou (47y, Consultant): You bring up an important point about isolation be...
